# Google Colab setup

## Install python version

In [ ]:
!sudo apt-get install python3.11 python3.11-distutils python3.11-dev

# 2. Register it as a 'python3' alternative (Priority 1)
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1

# 3. Force the system to use the new version
# This command bypasses the manual selection prompt
!sudo update-alternatives --set python3 /usr/bin/python3.11

# 4. Re-install pip for the new version
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --force-reinstall

# 5. Verify the change
!python3 --version

## Install requirements

In [ ]:
!pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install librosa
!pip install audioflux
!pip install pandas
!pip install matplotlib
!pip install ipykernel
!pip install ipympl
!pip install transformers
!pip install torchmetrics

## Mount drive

In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/Animacronica/SER-Experiments/'
sys.path.append(repo_path)

# Go to directory with repo
%cd {repo_path}

## Clone repo

In [ ]:
!git clone https://github.com/chrisalmonte/SER-Experiments.git

## Checkout a branch

In [ ]:
!git fetch origin
!git checkout Model-Training
!git branch

### Load Huggingface API key

In [ ]:
from google.colab import userdata
hf_key = userdata.get('Huggingface')

# Training
## Initialization

In [ ]:
#Imports
import math
import torch
import torchaudio
from torch import nn
from torch.utils.data import DataLoader
#from torchvision import transforms

#Custom modules
import cAudiotools
import cLogger
import cTransforms
import cModelManager
#import cNNModules

MODEL_NAME = "WavLM_B_VAD_baseline"
MODELS_DIR = "output/models/"
model_description = "WavLM Base for VAD regression using only output embeddings."

#Define output paths
log = cLogger.Log("output/logs", prefix=MODEL_NAME)
modelMngr = cModelManager.ModelManager(f"{MODELS_DIR}/{MODEL_NAME}")
log.log_property("model_name", MODEL_NAME)
log.log_property("model_description", model_description, show=False)
log.log_property("model_dir", modelMngr.model_directory, show=False)

#Torch properties and device
log.log_property("torch_version", torch.__version__)
log.log_property("torchaudio_version", torchaudio.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    log.log_property("device", "cuda")
    log.log_property("GPU_count", torch.cuda.device_count())
    log.log_property("GPU_device", torch.cuda.get_device_name(0))
else:
    log.log_property("device", "cpu")

## Define parameters

In [ ]:
#Parameters:
loader_params = {
    "dataset_train": r"data/labels_msp_consensus_undersampled.csv",
    "dataset_train_dir": r"C:/Datasets/_compiled/msp-podcast-2/Train",
    "dataset_dev": r"data/labels_dev_msp_consensus_undersampled.csv",
    "dataset_dev_dir": r"C:/Datasets/_compiled/msp-podcast-2/Development",
    "batch_size": 2,
    "shuffle": True,
    "collate_function": cAudiotools.Collate.waveform_dynamic_wMasks,
    "data_transform": None,
    "target_transform": cTransforms.NormalizeMinus(1, 7),
    "pin_memory": True,
    "num_workers": 0,
    "persistent_workers": False,
}
log.log_properties("Loader", loader_params, show=False)

training_params = {
    "epochs": 50,
    "checkpoint_interval": 5,
    "checkpoint_before_training": True,
    "criterion_for_best": "val_avg_loss",
}
log.log_properties("Training", training_params, show=False)

grad_acumulation_params = {
    "use_grad_accumulation": True,
    "simulated_batch_size": 16,
}
log.log_properties("Gradient Accumulation", grad_acumulation_params, show=False)

optimizer_params = {    
    "learning_rate": 1e-3,
    "adam_betas": (0.9, 0.999),
    "adam_epsilon": 1e-8,
}
log.log_properties("Optimizer", optimizer_params, show=False)

#scheduler_params = {
#    "use_scheduler": True,
#    "scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(),
#}

audio_params = {
    "sample_rate": 16000
}
log.log_properties("Audio", audio_params, show=False)

#spectrogram_params = {
#    "n_fft": 512,
#    "hop_length": 160,
#    "win_length": 400,
#    "window_fn": torch.hamming_window,
#    "normalized": True,
#    "power": 2
#}
#
#mel_params = {
#    "n_mels": 64,
#    "pad_mode": "constant",
#    "mel_scale": "htk",
#    "n_mfcc": 20
#}


## Create data loaders

In [ ]:

#spectogram_transform = torchaudio.transforms.Spectrogram(
#    n_fft=spectrogram_params["n_fft"],
#    hop_length=spectrogram_params["hop_length"],
#    win_length=spectrogram_params["win_length"],
#    window_fn=spectrogram_params["window_fn"],
#    normalized=spectrogram_params["normalized"],
#    power=spectrogram_params["power"]
#    )
#
#melspectogram_transform = torchaudio.transforms.MelSpectrogram(
#    sample_rate=audio_params["sample_rate"],
#    n_fft=spectrogram_params["n_fft"],
#    win_length=spectrogram_params["win_length"],
#    hop_length=spectrogram_params["hop_length"],
#    window_fn=spectrogram_params["window_fn"],
#    power=spectrogram_params["power"],
#    n_mels=mel_params["n_mels"],
#    normalized=spectrogram_params["normalized"],
#    pad_mode=mel_params["pad_mode"],
#    mel_scale=mel_params["mel_scale"]
#)
#
#mfcc_transform = torchaudio.transforms.MFCC(
#    sample_rate=audio_params["sample_rate"],
#    n_mfcc=mel_params["n_mfcc"],
#    melkwargs={
#        "n_fft": spectrogram_params["n_fft"],
#        "win_length": spectrogram_params["win_length"],
#        "hop_length": spectrogram_params["hop_length"],
#        "window_fn": spectrogram_params["window_fn"],
#        "power": spectrogram_params["power"],
#        "n_mels": mel_params["n_mels"],
#        "normalized": spectrogram_params["normalized"],
#        "pad_mode": mel_params["pad_mode"],
#        "mel_scale": mel_params["mel_scale"]
#    }
#)
#loader_params["data_transform"] = None

#log.log_properties("Spectrogram", spectrogram_params)
#log.log_properties("Mel Spectrogram", mel_params)

#Train set
dataset_train = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_train"], 
    loader_params["dataset_train_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
dataset_train_loader = DataLoader(
    dataset_train,
    batch_size=loader_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=loader_params["pin_memory"],
    num_workers=loader_params["num_workers"],
    persistent_workers=loader_params["persistent_workers"],
    )

#Development (validation) set
dataset_dev = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_dev"],
    loader_params["dataset_dev_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
dataset_dev_loader = DataLoader(
    dataset_dev,
    batch_size=loader_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=loader_params["pin_memory"],
    num_workers=loader_params["num_workers"],
    persistent_workers=loader_params["persistent_workers"],
    )


## Data check

In [ ]:
log.log_message(f"Train samples: {len(dataset_train)}")
log.log_message(f"Dev samples: {len(dataset_dev)}")
log.log_message(f"Batch size: {loader_params['batch_size']}")

sample_batch = next(iter(dataset_train_loader))
inputs, masks, targets = sample_batch

log.log_message("Sample batch:")
log.log_message(f"Inputs Shape: {inputs.shape}")
log.log_message(f"Targets Shape: {targets.shape}")
log.log_message(f"Masks Shape: {masks.shape}")
log.log_message(f"Input range: Min={inputs.min():.2f}, Max={inputs.max():.2f}")
log.log_message(f"Output range: Min={targets.min():.2f}, Max={targets.max():.2f}")

## Huggingface login

In [ ]:
from huggingface_hub import login
login(token=hf_key)

## Define Architecture

In [ ]:
from transformers import WavLMModel

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base")
        self.hidden_size = self.wavlm.config.hidden_size
        #Freeze weights
        self.wavlm.freeze_feature_encoder()
        for param in self.wavlm.parameters():
            param.requires_grad = False

        self.regression_head = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 3),
            nn.Tanh()
        )
    
    def frame_pooling(self, features, attention_masks):        
        if attention_masks is not None:
            feat_len = features.size(1)
            
            # Add dimensions for interpolation: (Batch, 1, Time)
            m = attention_masks.unsqueeze(1).float()
            
            # Downsample mask using Nearest Neighbor
            m = nn.functional.interpolate(m, size=feat_len, mode='nearest')
            
            # Transpose to match features: (Batch, Time, 1)
            m = m.transpose(1, 2)

            features = features * m
            pooled = features.sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = features.mean(dim=1)
            
        return pooled

    def forward(self, input, attention_masks):
        self.wavlm.eval()
        with torch.no_grad():
            ssl_output = self.wavlm(input, attention_mask=attention_masks)
            features = ssl_output.last_hidden_state  # (batch_size, seq_len, hidden_size)
        
        frame_pooled_embedding = self.frame_pooling(features, attention_masks)
        logits = self.regression_head(frame_pooled_embedding)
        return logits


model = NeuralNetwork().to(device)
log.log_property("model_structure", str(model))

loss_fn = nn.MSELoss()
log.log_property("loss_function", str(loss_fn))

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=optimizer_params["learning_rate"], 
    betas=optimizer_params["adam_betas"],
    eps=optimizer_params["adam_epsilon"]
    )

log.log_property("optimizer", str(optimizer))

## Training

In [ ]:
#Loop definitions
def train_loop(dataloader, model, loss_fn, optimizer, metrics_dict=None, pinned_memory=False):
    model.train()
    size = len(dataloader.dataset)
    epoch_loss = 0
    
    for batch, (inputs, masks, targets) in enumerate(dataloader):
        inputs = inputs.to(device, non_blocking=pinned_memory)
        targets = targets.to(device, non_blocking=pinned_memory)
        masks = masks.to(device, non_blocking=pinned_memory)

        pred = model(inputs, masks)
        loss = loss_fn(pred, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * inputs.size(0)
        batch_loss = loss.item()

        if batch % 100 == 0:
            current = batch * len(inputs)
            print(f"[{current:>6d}/{size:>6d}] batch loss: {batch_loss:>7f}")
    
    epoch_loss /= size
    if metrics_dict:
        metrics_dict["train_avg_loss"] = epoch_loss

#Loop definitions
def train_loop_grad_accumulation(dataloader, model, loss_fn, optimizer, batch_size, simulated_batch_size, metrics_dict=None, pinned_memory=False):
    model.train()
    size = len(dataloader.dataset)
    epoch_loss = 0
    optimizer.zero_grad(set_to_none=True)

    accumulation_steps = simulated_batch_size // batch_size
    
    for batch, (inputs, masks, targets) in enumerate(dataloader):
        inputs = inputs.to(device, non_blocking=pinned_memory)
        targets = targets.to(device, non_blocking=pinned_memory)
        masks = masks.to(device, non_blocking=pinned_memory)

        pred = model(inputs, masks)
        loss = loss_fn(pred, targets)
        #Normalize gradients
        loss = loss / accumulation_steps
        loss.backward()

        if (batch + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        #Print progress every 100 batches
        if batch % 100 == 0:
            current = batch * len(inputs)
            real_loss = loss.item() * accumulation_steps
            print(f"[{current:>6d}/{size:>6d}] batch loss: {real_loss:>7f}")

        epoch_loss += (loss.item() * accumulation_steps) * inputs.size(0)
    
    # Handle any remaining gradients if the dataset size isn't divisible by accumulation_steps
    if (batch + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    
    epoch_loss /= size
    if metrics_dict:
        metrics_dict["train_avg_loss"] = epoch_loss


def validation_loop(dataloader, model, loss_fn, metrics_dict=None, pinned_memory=False):
    model.eval()
    size = len(dataloader.dataset)
    test_loss = 0

    with torch.no_grad():
        for inputs, masks, targets  in dataloader:
            inputs = inputs.to(device, non_blocking=pinned_memory)
            targets = targets.to(device, non_blocking=pinned_memory)
            masks = masks.to(device, non_blocking=pinned_memory)

            pred = model(inputs, masks)
            loss = loss_fn(pred, targets)
            test_loss += loss.item() * inputs.size(0)

    test_loss /= size
    if metrics_dict:
        metrics_dict["val_avg_loss"] = test_loss


#Set Model Manager
modelMngr.set_model(model, optimizer, training_params["criterion_for_best"])

#Training
epoch_metrics = {'train_avg_loss': math.inf, 'val_avg_loss': math.inf}
remaining_for_checkpoint = training_params["checkpoint_interval"]

if training_params["checkpoint_before_training"]:
        modelMngr.checkpoint(0, epoch_metrics)
        log.save()

log.track_time(True, message="Starting training.")
total_epochs = training_params["epochs"]
pinned_memory = loader_params["pin_memory"]

for epoch in range(total_epochs):
    log.log_message(f"Epoch {epoch + 1} of {total_epochs}...")
    
    if grad_acumulation_params["use_grad_accumulation"]:
        train_loop_grad_accumulation(
            dataset_train_loader, model, loss_fn, optimizer, loader_params["batch_size"], grad_acumulation_params["simulated_batch_size"], 
            metrics_dict=epoch_metrics, pinned_memory=pinned_memory)
    else:
        train_loop(dataset_train_loader, model, loss_fn, optimizer, metrics_dict=epoch_metrics, pinned_memory=pinned_memory)
    
    validation_loop(dataset_dev_loader, model, loss_fn, metrics_dict=epoch_metrics, pinned_memory=pinned_memory)

    log.log_elapsed_time(message=f"Epoch {epoch + 1} completed.")
    log.log_epoch(epoch + 1, epoch_metrics)
    log.save()
    
    #Checkpointing
    remaining_for_checkpoint -= 1
    if remaining_for_checkpoint == 0:
        modelMngr.checkpoint(epoch + 1, epoch_metrics)
        remaining_for_checkpoint = training_params["checkpoint_interval"]
    
    #Save best model
    modelMngr.check_best(epoch + 1, epoch_metrics)

log.log_elapsed_time(message="Training completed")
log.track_time(False, show=False)
log.log_properties("Last_epoch", epoch_metrics)
log.log_properties("Best_model", modelMngr.best_model_metrics)
log.save()
#log.save_txt()

#Save last checkpoint if not saved at the end of training
if training_params["epochs"] % training_params["checkpoint_interval"] != 0:
    modelMngr.checkpoint(total_epochs, epoch_metrics)

modelMngr.save_for_inference()
modelMngr.save_best(save_for_inference=True)

## Evaluation

In [ ]:
from torchmetrics.regression import ConcordanceCorrCoef
from sklearn.metrics import mean_squared_error, mean_absolute_error

eval_params = {
    "dataset_test": r"C:/Datasets/_compiled/msp-podcast-2/labels_test1_VAD.csv",
    "dataset_test_dir": r"C:/Datasets/_compiled/msp-podcast-2/Test1",
    "shuffle": False,
    "collate_function": cAudiotools.Collate.waveform_dynamic_wLengths,
    "data_transform": None,
    "target_transform": cTransforms.NormalizeMinus(1, 7),
    "batch_size": 32,
    "pin_memory": True
}
log.log_properties("Evaluation", eval_params, show=False)

#Test set
dataset_test = cAudiotools.AudioDatasetVAD(
    eval_params["dataset_test"],
    eval_params["dataset_test_dir"],
    transform=eval_params["data_transform"],
    target_transform=eval_params["target_transform"]
    )
dataset_test_loader = DataLoader(
    dataset_test,
    batch_size=eval_params["batch_size"],
    shuffle=eval_params["shuffle"],
    collate_fn=eval_params["collate_function"],
    pin_memory=eval_params["pin_memory"]
)

log.log_message(f"Test samples: {len(dataset_test)}")
sample_batch = next(iter(dataset_test_loader))
inputs, masks, targets = sample_batch

log.log_message("Sample batch:")
log.log_message(f"Inputs Shape: {inputs.shape}")
log.log_message(f"Targets Shape: {targets.shape}")
log.log_message(f"Lengths Shape: {masks.shape}")
log.log_message(f"Input range: Min={inputs.min():.2f}, Max={inputs.max():.2f}")
log.log_message(f"Output range: Min={targets.min():.2f}, Max={targets.max():.2f}")

#Test loop
def test_loop(dataloader, model, device, pinned_memory=False):
    total_predictions = []
    total_targets = []
    model.eval()

    with torch.no_grad():
        print("Starting evaluation...")
        size = len(dataloader.dataset)
        for batch, (inputs, masks, targets) in enumerate(dataloader):
            inputs = inputs.to(device, non_blocking=pinned_memory)
            masks = masks.to(device, non_blocking=pinned_memory)

            pred = model(inputs, masks)
            total_predictions.append(pred.cpu())
            total_targets.append(targets)
            if batch % 10 == 0:
                current = batch * len(inputs)
                print(f"[{current:>6d}/{size:>6d}] processed")
        final_predictions = torch.cat(total_predictions, dim=0)
        final_targets = torch.cat(total_targets, dim=0)
        return final_predictions, final_targets

predictions, targets = test_loop(dataset_test_loader, model, device, pinned_memory=eval_params["pin_memory"])

log.log_message(f"Predictions Shape: {predictions.shape}")
log.log_message(f"Targets Shape: {targets.shape}")

concordance = ConcordanceCorrCoef(num_outputs=3)

results = {
    "Concordance_Correlation_Coefficient": concordance(predictions, targets).tolist(),
    "Mean_Squared_Error": mean_squared_error(targets.numpy(), predictions.numpy()),
    "Mean_Absolute_Error": mean_absolute_error(targets.numpy(), predictions.numpy())
}

log.log_properties("Test_results", results)
log.save()
log.save_txt()